In [109]:
!pip install qdrant-client -q
!pip install fastembed -q

In [110]:
import json
import collections
import pandas as pd
from typing import List

from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding, SparseTextEmbedding

### Load documents and ground truth data

In [43]:
with open('Data/data.json', 'rt') as f_in:
    documents = json.load(f_in)

In [42]:
gt_df =  pd.read_csv('ground_truth.csv')

### Qdrant client and collections


In [111]:
client = QdrantClient(":memory:")

In [132]:
def collection_exists(collection_name: str) -> bool:

    try:
        existing_collections = [col.name for col in client.get_collections().collections]
        return collection_name in existing_collections

    except:
        return False

In [133]:
def build_collection(name: str, vector_config: dict = None, sparse_vector_config: dict = None):

    try:
        vector_config = vector_config or {}
        sparse_vector_config = sparse_vector_config or {}

        exists = collection_exists(name)

        if exists:
            print(f"Collection '{name}' already exists.")
            return

        client.create_collection(
            collection_name = name,
            vectors_config = vector_config,
            sparse_vectors_config = sparse_vector_config
        )

        print(f"Qdrant collection '{name}' created.")

    except Exception as e:
        print(f"Failed to create collection '{name}': {e}")

In [134]:
def populate_collection(name: str, models_names: dict, documents: List[dict]):

    try:
        exists = collection_exists(name)

        if not exists:
            print(f"Collection '{name}' does not exists.")
            return

        points = []

        for record in documents:

            text_embed = f"{record['term']}: {record['definition']} {record['extra']}"

            dict_vector = {}

            for vector_name, model_name in models_names.items():
                dict_vector[vector_name] = models.Document(
                    text = text_embed,
                    model = model_name
                )

            point = models.PointStruct(
                id = record['id'],
                vector = dict_vector,
                payload = {
                    'term': record['term'],
                    'description': f"{record['definition']} {record['extra']}",
                    'models_used': models_names
                }
            )

            points.append(point)

        client.upsert(
            collection_name = name,
            points=points
        )

        print(f"Successfully populated collection '{name}' with {len(points)} records.")

    except Exception as e:
        print(f"An error occurred: {e}")

In [135]:
sample_data = [
    {
        'id': 1,
        'term': 'Artificial Intelligence',
        'definition': 'The simulation of human intelligence in machines that are programmed to think and learn.',
        'extra': 'Often abbreviated as AI; used in applications like chatbots and self-driving cars.'
    },
    {
        'id': 2,
        'term': 'Machine Learning',
        'definition': 'A subset of AI that involves training algorithms to learn from and make predictions on data.',
        'extra': 'Includes techniques like supervised and unsupervised learning.'
    },
    {
        'id': 3,
        'term': 'Neural Network',
        'definition': 'A series of algorithms that mimic the operations of a human brain to recognize relationships in data.',
        'extra': 'Forms the basis of deep learning.'
    },
    {
        'id': 4,
        'term': 'Overfitting',
        'definition': 'A modeling error that occurs when a function fits the training data too well.',
        'extra': 'Leads to poor performance on unseen data.'
    },
    {
        'id': 5,
        'term': 'Natural Language Processing',
        'definition': 'A field of AI focused on the interaction between computers and human languages.',
        'extra': 'Includes tasks like translation, sentiment analysis, and summarization.'
    },
    {
        'id': 6,
        'term': 'Computer Vision',
        'definition': 'A field of AI that trains computers to interpret and understand visual information.',
        'extra': 'Used in facial recognition and object detection.'
    },
    {
        'id': 7,
        'term': 'Gradient Descent',
        'definition': 'An optimization algorithm used to minimize the cost function in machine learning models.',
        'extra': 'Helps models learn by adjusting weights.'
    },
    {
        'id': 8,
        'term': 'Data Augmentation',
        'definition': 'Techniques used to increase the diversity of training data without collecting new data.',
        'extra': 'Common in image processing, like flipping or rotating images.'
    },
    {
        'id': 9,
        'term': 'Clustering',
        'definition': 'An unsupervised learning task that groups similar data points together.',
        'extra': 'K-Means is a popular clustering algorithm.'
    },
    {
        'id': 10,
        'term': 'Reinforcement Learning',
        'definition': 'An area of machine learning where agents learn to make decisions by receiving rewards or penalties.',
        'extra': 'Common in robotics and game playing (e.g., AlphaGo).'
    }
]

In [136]:
def semantic_search(question: str, collection_name: str, limit: int = 5):

    if not collection_exists(collection_name):

        print(f"Collection '{collection_name}' does not exist.")
        return

    vector_config = client.get_collection(collection_name).config.params.vectors

    if isinstance(vector_config, dict) and len(vector_config) > 1:

        print(f"Multiple dense embeddings used in collection '{collection_name}'. Cannot perform semantic search.")
        return

    sample_point = client.scroll(collection_name = collection_name, limit = 1)[0][0]

    used_models = sample_point.payload.get("models_used", {})

    if len(used_models) != 1:

        print(f"Collection '{collection_name}' must have exactly one model for semantic search.")
        return

    vector_name, model_name = next(iter(used_models.items()))

    query_doc = models.Document(text=question, model=model_name)

    results = client.query_points(
        collection_name = collection_name,
        query = query_doc,
        using = vector_name,
        limit = limit,
        with_payload = True
    )

    return [res.id for res in results.points]

In [137]:
def keyword_search(question: str, collection_name: str, limit: int = 5):

    if not collection_exists(collection_name):

        print(f"Collection '{collection_name}' does not exist.")
        return

    sparse_vector_config = client.get_collection(collection_name).config.params.sparse_vectors

    if isinstance(sparse_vector_config, dict) and len(sparse_vector_config) > 1:

        print(f"Multiple sparse embeddings used in collection '{collection_name}'. Cannot perform keyword search.")
        return

    sample_point = client.scroll(collection_name=collection_name, limit=1)[0][0]

    used_models = sample_point.payload.get("models_used", {})

    if len(used_models) != 1:

        print(f"Collection '{collection_name}' must have exactly one model for keyword search.")
        return

    vector_name, model_name = next(iter(used_models.items()))

    query_doc = models.Document(text=question, model=model_name)

    results = client.query_points(
        collection_name = collection_name,
        query = query_doc,
        using = vector_name,
        limit = limit,
        with_payload = True
    )

    return [res.id for res in results.points]

In [138]:
def multi_stage_search(question: str, collection_name: str, limit: int = 5):

    if not collection_exists(collection_name):

        print(f"Collection '{collection_name}' does not exist.")
        return

    vector_config = client.get_collection(collection_name).config.params.vectors

    sparse_vector_config = client.get_collection(collection_name).config.params.sparse_vectors

    if len(vector_config) + len(sparse_vector_config) < 1:

        print("Cannot perform multi-stage search.")
        return

    sample_point = client.scroll(collection_name=collection_name, limit=1)[0][0]

    used_models = sample_point.payload.get("models_used", {})

    dense = {}
    sparse = {}

    for vector_name, model_name in used_models.items():

        if vector_name in vector_config:
            dense = {"name": vector_name, "model": model_name}
        elif vector_name in sparse_vector_config:
            sparse = {"name": vector_name, "model": model_name}

    if not dense or not sparse:
        print("Both dense and sparse vectors are required for multi-stage search.")
        return

    results = client.query_points(
        collection_name = collection_name,
        query = models.Document(
            text = question,
            model = sparse["model"],
        ),
        using = sparse["name"],
        prefetch = [
            models.Prefetch(
                query = models.Document(
                    text = question,
                    model = dense["model"],
                ),
                using = dense["name"],
                limit = 2 * limit,
            )
        ],
        limit = limit,
        with_payload = True,
    )

    return [res.id for res in results.points]

In [139]:
def rrf_search(question: str, collection_name: str, limit: int = 5):

    if not collection_exists(collection_name):

        print(f"Collection '{collection_name}' does not exist.")
        return

    vector_config = client.get_collection(collection_name).config.params.vectors

    sparse_vector_config = client.get_collection(collection_name).config.params.sparse_vectors

    if len(vector_config) + len(sparse_vector_config) < 2:

        print("Both dense and sparse vectors are required for RRF search.")
        return

    sample_point = client.scroll(collection_name=collection_name, limit=1)[0][0]

    used_models = sample_point.payload.get("models_used", {})

    dense = {}
    sparse = {}

    for vector_name, model_name in used_models.items():

        if vector_name in vector_config:
            dense = {"name": vector_name, "model": model_name}
        elif vector_name in sparse_vector_config:
            sparse = {"name": vector_name, "model": model_name}

    if not dense or not sparse:

        print("Dense and sparse vector fields not found in the collection.")
        return

    results = client.query_points(
        collection_name = collection_name,
        query = models.FusionQuery(fusion=models.Fusion.RRF),
        prefetch = [
            models.Prefetch(
                query = models.Document(
                    text = question,
                    model = dense["model"],
                ),
                using = dense["name"],
                limit = 2 * limit,
            ),
            models.Prefetch(
                query = models.Document(
                    text = question,
                    model = sparse["model"],
                ),
                using = sparse["name"],
                limit = 2 * limit,
            ),
        ],
        limit = limit,
        with_payload = True,
    )

    return [res.id for res in results.points]


In [140]:
build_collection(
    name = 'Test1',
    sparse_vector_config = {
        'sparse_text': models.SparseVectorParams(
            modifier = models.Modifier.IDF
        )
    }
)
populate_collection(
    name='Test1',
    models_names={
        'sparse_text': 'Qdrant/bm25'
    },
    documents=sample_data
)

Collection 'Test1' already exists.
Successfully populated collection 'Test1' with 10 records.


In [141]:
keyword_search(question = 'Some AI, machine Learnig', collection_name='Test1')

[2, 1, 6, 7, 5]

In [142]:
build_collection(
    name = 'Test2',
    vector_config = {
        'dense_text': models.VectorParams(
            size = 512,
            distance = models.Distance.COSINE
        )
    }
)
populate_collection(
    name='Test2',
    models_names={
        'dense_text': 'jinaai/jina-embeddings-v2-small-en',
    },
    documents=sample_data
)

Collection 'Test2' already exists.
Successfully populated collection 'Test2' with 10 records.


In [143]:
semantic_search(question = 'Some AI, machine Learnig things.', collection_name='Test2')

[2, 1, 6, 5, 3]

In [144]:
build_collection(
    name = 'Test3',
    vector_config = {
        'dense_text': models.VectorParams(
            size = 512,
            distance = models.Distance.COSINE
        )
    },
    sparse_vector_config = {
        'sparse_text': models.SparseVectorParams(
            modifier = models.Modifier.IDF
        )
    }
)
populate_collection(
    name='Test3',
    models_names={
        'dense_text': 'jinaai/jina-embeddings-v2-small-en',
        'sparse_text': 'Qdrant/bm25'
    },
    documents=sample_data
)

Collection 'Test3' already exists.
Successfully populated collection 'Test3' with 10 records.


In [145]:
multi_stage_search(question = "What Artificial, Machine going on here?", collection_name = 'Test3')

[1, 7, 2, 10]

In [146]:
rrf_search(question = "What Artificial, Machine going on here?", collection_name = 'Test3')

[1, 7, 2, 10, 6]

In [147]:
def score_hit_rate(output):

    hits = 0
    for id, document_ids in output.items():
        for document_id in document_ids:
            for doc_id in document_id:
                if id == doc_id:
                    hits += 1
                    break

    return hits

def score_mrr(ouput):

    score = 0.0

    for id, document_ids in output.items():
        for document_id in document_ids:
            for index, doc_id in enumerate(document_id):
                if id == doc_id:
                    score += (1/(index+1))
                    break

    return score